# 1. Define Data And Snapshot

Explore the raw Home Credit source, make durable project decisions in `config.py`, preview the configured pipeline, then create a logged immutable dataset.

## 1. Resolve Project Folder

Select the project folder from the notebook location. This does not require the project config to be final.

In [1]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

import automl
from automl import data
from automl.data import FeatureRegistry
from automl.data.sources import LocalCSVSource

DRY_RUN = True


## 2. Choose A Candidate Source

Supported source patterns are ordinary Python objects. Keep one active source for the preview.

```python
# Local CSV
# source = LocalCSVSource(csv_path=config.project_dir / "data/application_train_sample.csv", hash_key="SK_ID_CURR")

# GCS parquet
# source = GCSParquetSource(gcs_uri="gs://bucket/path/application_train.parquet", hash_key="SK_ID_CURR")

# Snowflake query
# source = SnowflakeSource(base_table="DB.SCHEMA.TABLE", base_data_sql="data/queries/base_data.sql", training_data_sql="data/queries/training_data.sql")
```

In [2]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)


{'project': 'example_homecredit',
 'repo_root': '/Users/wendao/1.working_directory/automl',
 'project_dir': '/Users/wendao/1.working_directory/automl/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

## 3. Inspect Raw Columns

Look at raw columns before committing project config decisions.

In [3]:
source = LocalCSVSource(
    csv_path=config.project_dir / "data" / "application_train_sample.csv",
    hash_key="SK_ID_CURR",
)
source_df = source.load(project_dir=config.project_dir, nrows=1000)
source_df.head()


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
raw_overview = {
    "shape": source_df.shape,
    "dtypes": source_df.dtypes.astype(str).to_frame("dtype"),
    "missingness": source_df.isna().mean().sort_values(ascending=False).to_frame("null_rate"),
    "target_distribution": source_df["TARGET"].value_counts(dropna=False).to_frame("rows"),
}
raw_overview


{'shape': (1000, 122),
 'dtypes':                               dtype
 SK_ID_CURR                    int64
 TARGET                        int64
 NAME_CONTRACT_TYPE           object
 CODE_GENDER                  object
 FLAG_OWN_CAR                 object
 ...                             ...
 AMT_REQ_CREDIT_BUREAU_DAY   float64
 AMT_REQ_CREDIT_BUREAU_WEEK  float64
 AMT_REQ_CREDIT_BUREAU_MON   float64
 AMT_REQ_CREDIT_BUREAU_QRT   float64
 AMT_REQ_CREDIT_BUREAU_YEAR  float64
 
 [122 rows x 1 columns],
 'missingness':                           null_rate
 COMMONAREA_MODE               0.697
 COMMONAREA_AVG                0.697
 COMMONAREA_MEDI               0.697
 NONLIVINGAPARTMENTS_MODE      0.689
 NONLIVINGAPARTMENTS_AVG       0.689
 ...                             ...
 NAME_FAMILY_STATUS            0.000
 NAME_EDUCATION_TYPE           0.000
 NAME_INCOME_TYPE              0.000
 AMT_ANNUITY                   0.000
 SK_ID_CURR                    0.000
 
 [122 rows x 1 columns],
 'target_d

In [5]:
hash_key_cols = ["SK_ID_CURR"]
pd.Series(
    {
        "hash_key": hash_key_cols,
        "duplicate_key_rows": int(source_df.duplicated(subset=hash_key_cols, keep=False).sum()),
        "null_key_rows": int(source_df[hash_key_cols].isna().any(axis=1).sum()),
    }
)


hash_key              [SK_ID_CURR]
duplicate_key_rows               0
null_key_rows                    0
dtype: object

## 4. Draft Feature Registry Decisions

Use the registry interactively to review target, metadata, excluded columns, and initial feature roles before editing `config.py`.

In [6]:
metadata_candidates = [col for col in source_df.columns if "ID" in col.upper() or col.upper().endswith("SK_ID_CURR")]
metadata_candidates


['SK_ID_CURR', 'DAYS_ID_PUBLISH']

## 5. Edit `config.py`

Open and edit the durable project definition directly. Do not generate config source from the notebook.

Review these objects in `config.py`:

- `TASK`
- `DATA`
- `EVAL`
- `RUN_CONFIG`
- source block
- metadata and excluded columns
- dry-run row limit/default dry-run routing

In [7]:
data_spec = config.require_data_spec()
registry = FeatureRegistry().build_from_df(
    source_df,
    target_column=config.raw_target_column,
    metadata_cols=data_spec.metadata_cols,
    exclude_cols=data_spec.exclude_cols,
)
registry.add_comment("SK_ID_CURR", "metadata: application identifier, not a model feature")
registry.to_dataframe()


,name,dtype,original_name,null_pct,nunique,dominance_pct,available,feature,model,target,comments,derived,source_columns
0,SK_ID_CURR,num,SK_ID_CURR,0.000,1000,0.001,True,False,False,False,"metadata: application identifier, not a model ...",False,[]
1,TARGET,num,TARGET,0.000,2,0.930,True,False,False,True,,False,[]
2,NAME_CONTRACT_TYPE,cat,NAME_CONTRACT_TYPE,0.000,2,0.903,True,True,True,False,,False,[]
3,CODE_GENDER,cat,CODE_GENDER,0.000,2,0.644,True,True,True,False,,False,[]
4,FLAG_OWN_CAR,cat,FLAG_OWN_CAR,0.000,2,0.665,True,True,True,False,,False,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,num,AMT_REQ_CREDIT_BUREAU_DAY,0.144,3,0.852,True,True,True,False,,False,[]
118,AMT_REQ_CREDIT_BUREAU_WEEK,num,AMT_REQ_CREDIT_BUREAU_WEEK,0.144,3,0.827,True,True,True,False,,False,[]
119,AMT_REQ_CREDIT_BUREAU_MON,num,AMT_REQ_CREDIT_BUREAU_MON,0.144,9,0.729,True,True,True,False,,False,[]
120,AMT_REQ_CREDIT_BUREAU_QRT,num,AMT_REQ_CREDIT_BUREAU_QRT,0.144,5,0.678,True,True,True,False,,False,[]


## 6. Load Config And Preview Pipeline Output

After editing `config.py`, load the typed project and preview the configured pipeline locally. Previewing does not log MLflow or GCS artifacts.

In [8]:
config.config_path


PosixPath('/Users/wendao/1.working_directory/automl/projects/example_homecredit/config.py')

## 7. Explore Previewed Pipeline Data

Review the post-config train/test split view, registry, split report, target distribution, and normalized dtypes. Loop back to `config.py` if anything is wrong.

In [9]:
loaded = data.build_dataset(session=active)
run_config = active.config.require_run_config()
train_buckets = run_config.splits.buckets(run_config.train_split)
holdout_buckets = run_config.splits.buckets(run_config.eval_split)
train_preview = loaded.df[loaded.df[loaded.dataset.split_id_col].isin(train_buckets)].reset_index(drop=True)
holdout_preview = loaded.df[loaded.df[loaded.dataset.split_id_col].isin(holdout_buckets)].reset_index(drop=True)
loaded


LoadedDataset(dataset=Dataset(id='unmaterialized', identity_hash='sha256:51b1c03a9d267c7e01926685057a82990b9a1c74fb4759eda3f9b77d8875f1fd', component_hashes=ComponentHashes(source_identity='sha256:11d9b27b6304ded71c9fee78bd717d839a2bb40e27f34fff6b31461a93b39b02', feature_registry='sha256:5153ea24d069ff3a97e483ec7891b27caa3101eeeda4e9647d6ad5c7c7a31255', data_content='sha256:dcdc36389dbfe7a30b130ebe81cd4837cc04ac388ddadf484b07136c6a6591d4', schema='sha256:8b7a337cb399ab0d2b66984a0882bcabc8783f321b0f581fc1535386a4b8e380'), gcs_bucket='data_science_test_remote', project_name='example_homecredit', created_at='2026-06-02T17:02:50.971558+00:00', source_identity={'kind': 'local_csv', 'csv_path': '/Users/wendao/1.working_directory/automl/projects/example_homecredit/data/application_train_sample.csv', 'hash_key': ['sk_id_curr']}, n_rows=100, n_columns=108, target_column='target', split_id_col='SPLITID', hash_key=('sk_id_curr',), gcs_prefix='automl/dry_run', schema_version=1), df=    sk_id_curr 

In [10]:
train_preview.head()


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_14,flag_document_16,flag_document_18,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,SPLITID
0,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,62
1,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3
2,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,41
3,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,74
4,100009,0,Cash loans,F,Y,Y,1,171000.0,1560726.0,41301.0,...,1,0,0,0.0,0.0,0.0,1.0,1.0,2.0,35


In [11]:
holdout_preview.head()


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_14,flag_document_16,flag_document_18,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,SPLITID
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,85
1,100008,0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,90
2,100010,0,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,87
3,100021,0,Revolving loans,F,N,Y,1,81000.0,270000.0,13500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,82
4,100022,0,Revolving loans,F,N,Y,0,112500.0,157500.0,7875.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,89


In [12]:
loaded.registry.to_dataframe()


,name,dtype,original_name,null_pct,nunique,dominance_pct,available,feature,model,target,comments,derived,source_columns
0,sk_id_curr,num,SK_ID_CURR,0.00,100,0.01,True,False,False,False,,False,[]
1,target,num,TARGET,0.00,2,0.94,True,False,False,True,,False,[]
2,name_contract_type,cat,NAME_CONTRACT_TYPE,0.00,2,0.85,True,True,True,False,,False,[]
3,code_gender,cat,CODE_GENDER,0.00,2,0.57,True,True,True,False,,False,[]
4,flag_own_car,cat,FLAG_OWN_CAR,0.00,2,0.70,True,True,True,False,,False,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,amt_req_credit_bureau_week,num,AMT_REQ_CREDIT_BUREAU_WEEK,0.15,2,0.83,True,True,True,False,,False,[]
104,amt_req_credit_bureau_mon,num,AMT_REQ_CREDIT_BUREAU_MON,0.15,4,0.70,True,True,True,False,,False,[]
105,amt_req_credit_bureau_qrt,num,AMT_REQ_CREDIT_BUREAU_QRT,0.15,4,0.69,True,True,True,False,,False,[]
106,amt_req_credit_bureau_year,num,AMT_REQ_CREDIT_BUREAU_YEAR,0.15,7,0.31,True,True,True,False,,False,[]


In [13]:
{
    "train_split": run_config.train_split,
    "train_buckets": sorted(train_buckets),
    "eval_split": run_config.eval_split,
    "eval_buckets": sorted(holdout_buckets),
    "split_id_col": loaded.dataset.split_id_col,
}


{'train_split': 'train',
 'train_buckets': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79],
 'eval_split': 'test',
 'eval_buckets': [80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99],
 'split_id_col': 'SPLITID'}

In [14]:
train_preview[loaded.dataset.target_column].value_counts(dropna=False).to_frame("train_rows")


,train_rows
target,
0,75
1,4


## 8. Create Logged Dataset

Run this only when the previewed pipeline matches the intended contract. This writes the full immutable dataset parquet, feature registry, data manifest, dataset index, active dataset pointer, and source trace artifacts.

In [15]:
loaded.df.dtypes.astype(str).to_frame("dtype")


,dtype
sk_id_curr,int64
target,int64
name_contract_type,object
code_gender,object
flag_own_car,object
...,...
amt_req_credit_bureau_week,float64
amt_req_credit_bureau_mon,float64
amt_req_credit_bureau_qrt,float64
amt_req_credit_bureau_year,float64


## 9. Inspect Snapshot Result

Record the stable IDs and artifact pointers. Later notebooks should reload these artifacts instead of rebuilding the pipeline.

In [16]:
dataset = data.materialize(session=active)
materialized = dataset.dataset
dataset_summary = {
    "dataset_id": materialized.id,
    "identity_hash": materialized.identity_hash,
    "data_uri": materialized.data_gcs_uri,
    "feature_registry_uri": materialized.registry_gcs_uri,
    "manifest_uri": materialized.manifest_gcs_uri,
    "source_identity_hash": materialized.component_hashes.source_identity,
    "dataset_content_hash": materialized.component_hashes.data_content,
    "schema_hash": materialized.component_hashes.schema,
}
dataset_summary


🏃 View run overview at: https://blue.hellobrigit.com/#/experiments/23/runs/bd2f1622293c4a248f8de991927d852a
🧪 View experiment at: https://blue.hellobrigit.com/#/experiments/23


{'dataset_id': 'v1_51b1c03a',
 'identity_hash': 'sha256:51b1c03a9d267c7e01926685057a82990b9a1c74fb4759eda3f9b77d8875f1fd',
 'data_uri': 'gs://data_science_test_remote/automl/dry_run/example_homecredit/data/datasets/v1_51b1c03a/data.parquet',
 'feature_registry_uri': 'gs://data_science_test_remote/automl/dry_run/example_homecredit/data/datasets/v1_51b1c03a/feature_registry.csv',
 'manifest_uri': 'gs://data_science_test_remote/automl/dry_run/example_homecredit/data/datasets/v1_51b1c03a/manifest.json',
 'source_identity_hash': 'sha256:11d9b27b6304ded71c9fee78bd717d839a2bb40e27f34fff6b31461a93b39b02',
 'dataset_content_hash': 'sha256:dcdc36389dbfe7a30b130ebe81cd4837cc04ac388ddadf484b07136c6a6591d4',
 'schema_hash': 'sha256:8b7a337cb399ab0d2b66984a0882bcabc8783f321b0f581fc1535386a4b8e380'}